# ReTone — Direct Instrument Conversion Pipeline

> Take any audio recording. Transcribe it. Re-render it as a different instrument.

This notebook documents the **direct pipeline** — our primary approach for change-instrument rendering. It uses no neural network at inference time; only Basic Pitch (or ByteDance for piano) at the transcription stage, followed by MIDI arrangement and soundfont-based synthesis.

**Source code**: [`training/direct/`](../training/direct/) — `instruments.py`, `arrange.py`, `render_direct.py`.

## Table of contents

1. [Architecture](#1-architecture)
2. [Why direct beats our ML approach](#2-why-direct-beats-our-ml-approach)
3. [Installation](#3-installation)
4. [Quick start](#4-quick-start-single-render)
5. [The instrument catalog](#5-the-instrument-catalog)
6. [Batch rendering](#6-batch-rendering)
7. [Transcription: Basic Pitch vs ByteDance](#7-transcription-basic-pitch-vs-bytedance-pianotranscription)
8. [Arrangement stages](#8-arrangement-stages)
9. [Adding new instruments / soundfonts](#9-adding-new-instruments--soundfonts)
10. [The ML approach we tried — and its shortcomings](#10-the-ml-approach-we-tried--and-its-shortcomings)
11. [Roadmap](#11-roadmap--improvements)

## 1. Architecture

```
  input.wav
      │
      ▼
  ┌─────────────────────────────────┐
  │ Stage 1  ─ Transcribe (BP / ByteDance) │   ───▶  pretty_midi.PrettyMIDI  (notes)
  └─────────────────────────────────┘
      │
      ▼
  ┌─────────────────────────────────┐
  │ Stage 2  ─ Arrange per target        │   ───▶  MIDI (extended, legato-fixed)
  │            • piano_sustain             │
  │            • strings sostenuto         │
  │            • none (perc / pizz)        │
  └─────────────────────────────────┘
      │
      ▼
  ┌─────────────────────────────────┐
  │ Stage 3  ─ FluidSynth(SF2, program)  │   ───▶  dry WAV @ 44.1 kHz
  └─────────────────────────────────┘
      │
      ▼
  ┌─────────────────────────────────┐
  │ Stage 4  ─ Light hall reverb (15%)   │   ───▶  output.wav
  └─────────────────────────────────┘
```

**Everything is deterministic.** Given the same input audio and the same catalog entry, the output is byte-for-byte reproducible.

**Runtime**: ~5–25 s per song depending on transcriber (Basic Pitch on CPU: ~10 s for 15 s of audio; ByteDance on GPU: ~2 s). The FluidSynth and reverb stages together are <2 s.

## 2. Why direct beats our ML approach

We spent a couple of days training a `PianoRollToMel` model that maps transcribed piano roll → mel spectrogram, then decodes the mel with a frozen BigVGAN vocoder. It worked (validation loss dropped from 1.0 to ~0.12), but audible A/B testing revealed:

> "The raw SF2 render (target the model was trying to imitate) sounds much better than the model output."

The model is bounded above by "what its training data sounds like" and further degraded by the neural mel→wav vocoder. So the model was a **lossy middleman** between the transcribed MIDI and the sound we actually wanted.

| dimension                        | ML pipeline (`training/poly/`)        | Direct pipeline (this notebook) |
|----------------------------------|---------------------------------------|---------------------------------|
| inference latency                | ~10–40 s (transcribe + PianoRollToMel + BigVGAN) | ~5–25 s (transcribe + FluidSynth) |
| VRAM at inference                | 10–12 GB                              | 0 (CPU-only render)             |
| timbre fidelity                  | bounded by mel + BigVGAN losses       | perfect SF2 sample playback     |
| training cost                    | ~$50–100 per instrument on A40         | zero                            |
| new-instrument onboarding        | \~5 h training + tuning               | drop an SF2 file, one line in `instruments.py` |
| generalises to unseen note combos | only if in training distribution      | fluidsynth plays anything valid |
| expressive nuance from source    | destroyed by transcriber              | destroyed by transcriber (same problem) |

The ML approach still has a future purpose — capturing performance nuances a soundfont cannot express (room feel, per-note dynamics, vibrato). But that requires a fundamentally different architecture (loudness/vibrato conditioning + higher-res mel + adversarial loss), not the mel-imitation we built. See [§10](#10-the-ml-approach-we-tried--and-its-shortcomings) for the postmortem.

## 3. Installation

### System packages

```bash
# macOS
brew install fluidsynth ffmpeg

# Ubuntu / RunPod pod
apt-get install -y fluidsynth fluid-soundfont-gm ffmpeg unzip
```

### Python

```bash
pip install librosa soundfile pretty_midi scipy basic-pitch

# Optional — for piano-specific transcription (much higher F1)
pip install piano_transcription_inference
```

### Soundfonts

**FluidR3 GM** (128 GM programs, ~150 MB, MIT-licensed) comes with `fluid-soundfont-gm` on Debian/Ubuntu at `/usr/share/sounds/sf2/FluidR3_GM.sf2`. On macOS, `brew install fluid-synth` doesn't ship a soundfont; download from [Musical Artifacts](https://musical-artifacts.com/artifacts/738) and set the path in `instruments.py`.

**Sonatina Symphonic Orchestra** (~500 MB unzipped, 58 per-instrument SF2s) — unlocks real sampled violins/violas/cellos/brass/woodwinds:

```bash
mkdir -p /workspace/sf2 && cd /workspace/sf2
curl -sL -o sonatina.zip \
  "https://archive.org/download/SonatinaSymphonicOrchestraSF2/Sonatina%20Symphonic%20Orchestra%20SF2.zip"
unzip -q sonatina.zip && rm sonatina.zip
```

Then update `SONATINA_DIR` in [`instruments.py`](../training/direct/instruments.py) if you put it somewhere other than `/workspace/sf2`.

## 4. Quick start: single render

Convert an audio file into any instrument in the catalog.

In [ ]:
!python ../training/direct/render_direct.py \
    --input  ../test_audio/bohemian_piano.wav \
    --instrument  cello_sustain \
    --out  /tmp/bohemian_as_cello.wav

Or from Python — the library-style API:

In [ ]:
import sys; sys.path.insert(0, '../training/direct')
from render_direct import render_one

render_one(
    input_audio='../test_audio/bohemian_piano.wav',
    instrument_name='cello_sustain',
    out_wav='/tmp/bohemian_as_cello.wav',
    transcriber='basic_pitch',           # or 'bytedance_piano' for piano source
    seconds=12,                          # clip to first 12 s (optional)
    reverb_wet=0.15,                     # 0.0 dry, 0.3 wet
)

## 5. The instrument catalog

All available instruments live in [`training/direct/instruments.py`](../training/direct/instruments.py) as an `INSTRUMENTS` dict of `Instrument(name, display, category, sf2, program, arranger)`.

As of this notebook the catalog ships ~65 entries across 10 categories: piano, keys, organ, guitar, bass, strings, brass, wind, voice, harp, synth.

Browse the full catalog:

In [ ]:
!python ../training/direct/render_direct.py --list

Or programmatically:

In [ ]:
from instruments import INSTRUMENTS, by_category, list_categories

print('categories:', list_categories())
print()
for i in by_category('piano'):
    print(f"  {i.name:32s} {i.display:32s} (program {i.program} in {i.sf2.split('/')[-1]})")

### Highlights

**Pianos (7 variants)** — grand, bright, electric grand, honky-tonk, Rhodes-style EP1, chorused EP2, and the real-sampled Sonatina Grand Piano.

**Strings (~20 variants)** — FluidR3 GM's synthetic violins/violas/cellos/contrabass + all Sonatina real-sampled sections (1st Violins Sustain/Pizzicato/Staccato, 2nd Violins Sustain, Violin Solo, Violas Sustain/Pizz, Celli Sustain/Pizz, Cello Solo, Basses Sustain/Pizz).

**Brass (9)** — FluidR3 trumpet/trombone/tuba/horn/ensemble + Sonatina Trumpet Solo, Horns Sustain, Trombones Sustain, Tuba Sustain.

**Woodwinds (9)** — FluidR3 sax family + flute/oboe/clarinet + Sonatina solo variants of flute/oboe/clarinet/bassoon.

**Guitars (6)** — nylon, steel, jazz, clean, overdrive, distortion.

**Voice / choir / harp / synths** — also covered.

See [§9](#9-adding-new-instruments--soundfonts) to add more.

## 6. Batch rendering

Render one input through every instrument of a category — useful for A/B previews ("give me my song as all 20 strings variants"):

In [ ]:
!python ../training/direct/render_direct.py \
    --input  ../test_audio/song.wav \
    --category  strings \
    --outdir  /tmp/strings_variants/

The output files land as `{input_stem}__{instrument_name}.wav` per instrument.

## 7. Transcription: choosing a transcriber

Default is Basic Pitch. For piano-source audio we ship a ByteDance option. Here is the current (late 2025) landscape of pip-installable transcribers, ranked by F1:

| Transcriber                          | Best for                | Note-F1 (MAESTRO) | License      | Install                                    |
|--------------------------------------|-------------------------|-------------------|--------------|--------------------------------------------|
| **Basic Pitch** (Spotify, 2022)      | Any polyphonic          | ~85               | Apache-2.0   | `pip install basic-pitch`                  |
| **ByteDance PianoTranscription** ('20)| Piano                  | 96.72             | Apache-2.0   | `pip install piano_transcription_inference`|
| **Transkun** (Yan, ISMIR 2024)       | Piano (clean studio)    | ~97.5+            | MIT          | `pip install transkun`                     |
| **Aria-AMT** (EleutherAI, 2024)      | Piano (noisy real-world)| ≈ Transkun, more robust | Apache-2.0 | source install + weights from [HF](https://huggingface.co/loubb/aria-medium-base) |
| **mt3-infer** (openmirlab, 2025)     | Multi-instrument mixed  | multi-stem via YourMT3+ backend | MIT | `pip install mt3-infer[torch]`             |

**Community best practice** for a production pipeline: **separate first, then transcribe per stem.** BS-Roformer ([Music-Source-Separation-Training](https://github.com/ZFTurbo/Music-Source-Separation-Training)) splits mixes into piano/vocal/bass/drums, then each stem routes to its best transcriber (Transkun for piano, drum-AMT for drums, Basic Pitch for everything else). See [mason369/music-to-midi](https://github.com/mason369/music-to-midi) for a working reference implementation with 13 selectable routes.

The shipped ByteDance path (currently the piano upgrade users can invoke directly):

In [ ]:
!pip install -q piano_transcription_inference

!python ../training/direct/render_direct.py \
    --input  ../test_audio/bohemian_piano.wav \
    --instrument  cello_sustain \
    --transcriber  bytedance_piano \
    --out  /tmp/bohemian_as_cello_bytedance.wav

### Extending: adding a third transcriber

The transcriber dispatch is a plain Python dict in `render_direct.py`. To add MT3, hFT-Transformer, or your own:

```python
def transcribe_mt3(audio_path):
    # ...
    return pretty_midi_object

TRANSCRIBERS['mt3'] = transcribe_mt3
```

Any transcriber must return a `pretty_midi.PrettyMIDI` object. Everything downstream (arrange, render, reverb) is transcriber-agnostic.

## 8. Arrangement stages

Between transcribe and render, we apply a per-target-instrument MIDI transform. Three options, selected by the `arranger` field on each catalog entry:

**`piano_sustain`** — for percussive-family targets (piano, keys, guitar, marimba, harp).

Extends each already-substantive note (>100 ms) by ~300 ms so the render sounds pedaled rather than choppy. Transcribers strip real CC64 pedal info; this partially recovers it.

**`strings`** — for bowed-family targets (strings, brass, wind, choir).

Sostenuto extension: each held note's end is stretched to the next onset in its voice (nearest neighbor within ±3 semitones), capped at +0.8 s, with 40 ms legato overlap. Skips notes shorter than 200 ms so staccato lines stay staccato.

**`none`** — for plucked / staccato / percussive-attack targets (pizzicato variants, percussion).

MIDI passes through unchanged. Notes should stay short.

See [`training/direct/arrange.py`](../training/direct/arrange.py). All three functions live in `ARRANGERS = {...}` — add your own by appending to that dict.

In [ ]:
# Compare arrangement effects on the same transcript
from arrange import arrange_for_piano_sustain, arrange_for_strings
import pretty_midi, copy

pm = pretty_midi.PrettyMIDI('/tmp/some_transcription.mid')     # from any transcriber
n_before = sum(len(i.notes) for i in pm.instruments)
ends_before = sorted(round(n.end, 3) for i in pm.instruments for n in i.notes)[:5]

pm_p = arrange_for_piano_sustain(copy.deepcopy(pm))
pm_s = arrange_for_strings(copy.deepcopy(pm))

print(f'orig  : {n_before} notes, first-5 ends {ends_before}')
print(f'piano : {sum(len(i.notes) for i in pm_p.instruments)} notes, extended by ~0.3 s each')
print(f'strings: {sum(len(i.notes) for i in pm_s.instruments)} notes, held to next onset')

## 9. Adding new instruments / soundfonts

Two steps:

1. **Drop the SF2 or SFZ file somewhere on disk.** For SFZ format you'll need `sfizz` instead of FluidSynth — not covered here but the same pattern applies.
2. **Append a row to `instruments.py`.**

```python
# in _ENTRIES:
("my_koto",  "Koto (My SF2)",  "strings",
  "/path/to/koto.sf2",  0,  "strings"),
```

That's it. No re-training, no cache rebuild, no server restart. Next invocation of `render_direct.py --list` shows it; `--instrument my_koto` uses it.

### Where to find good SF2s (all license-clean for research/commercial pending audit)

| Source                                | Contents                                        | License                          | Size    |
|---------------------------------------|-------------------------------------------------|----------------------------------|---------|
| FluidR3 GM                            | 128 GM programs (basic but complete)            | MIT                              | 150 MB  |
| Sonatina Symphonic Orchestra          | 58 orchestral instruments                       | CC-Sampling+ 1.0                 | 500 MB  |
| GeneralUser GS                        | Alt 128 GM (fuller sound than FluidR3)          | permissive (check LICENSE)       | 30 MB   |
| Salamander Grand Piano V3             | Real Yamaha C5, 16 velocity layers              | CC-BY 3.0 (credit Alexander Holm) | 800 MB  |
| VSCO 2 Community Edition              | Full orchestral SFZ (needs sfizz)               | CC-BY 4.0                        | 9 GB    |
| [MuseScore Soundfonts](https://github.com/spessasus/SoundFonts)   | Community collection                            | Various, per-file                | Varies  |

For SFZ (instead of SF2), the same catalog structure works — just swap the FluidSynth call in `render_direct.py::render_sf2` for `sfizz_render`. sfizz is available as `pip install sfizz` on Linux.

## 10. The ML approach we tried — and its shortcomings

For posterity. Everything below refers to [`training/poly/`](../training/poly/) which is now deprecated as the primary path but preserved as reference and future-work seed material.

### What we built

```
audio → Basic Pitch → piano_roll [3×128×T]
                          ↓
                    PianoRollToMel        ←— the 64 M-param Transformer we trained
                          ↓
                       mel [128×T]
                          ↓
                       BigVGAN v2 44 kHz  ←— frozen; nvidia/bigvgan_v2_44khz_128band_512x
                          ↓
                      output.wav
```

**Data**: MAESTRO MIDI → FluidSynth(target SF2) → 60 s WAVs → mel; MIDI → piano roll (`3×128×T`: onset ramp, sustain, velocity). Cached 260–1500 pairs per target instrument.

**Model**: Conv1d stem → PositionalEncoding → TransformerEncoder(8 × 768, 12 heads) → Conv1d head. ~64 M params. Trained bf16 mixed precision on an A40 48 GB.

**Loss**: L1(pred_mel, target_mel) + 0.5 · L1(Δpred, Δtarget). Delta term encourages temporal smoothness.

**Training tricks we added over 24 h**:
- Mel-domain augmentation (level ±0.35, spectral tilt, noise floor, SpecAugment, velocity jitter)
- Real MAESTRO Disklavier audio mixed into piano cache (fetched via `remotezip` from the 108 GB archive, no monolith download)
- Warm-start plumbing (per-instrument fine-tune from a shared strings checkpoint, val 0.116)
- Round-robin rotator across 8 instruments, 2500 steps per turn

**Best validation losses reached**:
- Piano (synth): 0.119
- Piano (mixed real+synth): 0.163
- Strings ensemble (FluidR3): 0.116
- New instruments (violin/viola/cello Sonatina): 0.114–0.177 after few thousand steps

### Where it fell short

**1. Bounded above by the target audio.** The model is trained to reproduce a FluidSynth render. On subjective A/B, listeners consistently prefer the FluidSynth render itself. Val 0.11 means "very close to FluidSynth" but the neural mel→wav path adds artifacts that never quite disappear.

**2. The mel bottleneck is lossy.** 128 mel bands at 512-sample hop at 44.1 kHz — each band spans ~172 Hz. Strings have harmonic detail above that spacing; piano attack transients too. Sparse-harmonic instruments (piano) survive better than dense-harmonic ones (strings). Explains why our piano output was almost-usable while strings sounded thin.

**3. Averaging over training examples.** Deterministic model with L1 loss returns the mean over the training-target distribution for any input. Result: bland, smooth outputs. Vibrato, bow noise, string squeak, mic pre color — all averaged away.

**4. One-soundfont overfitting was real.** We fought it with augmentation and multi-cache mixing (later reverted per user's "one instrument, one training" call) but the residual is audible: the model sounds like a soft-focus FluidSynth.

**5. High training cost per instrument.** ~5 h GPU per instrument to warm-start and refine. For our final 8-instrument lineup that's ~40 GPU-hours (~$40–100). Direct pipeline: zero.

**6. Onboarding new instruments is slow.** Every new SF2 requires a render → cache → train cycle. Direct pipeline: add one row.

### What the ML approach could still contribute (future work)

**Performance nuance capture** — a redesigned model that takes not just note events but the source audio's f0 curve, RMS envelope, and spectral centroid as conditioning. That would let it reproduce vibrato, dynamics, and the *feel* of the source performance, which the direct pipeline cannot. Different architecture (probably diffusion or flow-matching in a codec latent, à la AFTER), different training, different notebook — not this one.

The checkpoints from our attempt are preserved on Hugging Face at [Ganeshveeer/retone-poly](https://huggingface.co/Ganeshveeer/retone-poly) as reference / warm-start material.

## 11. Roadmap / improvements

Ranked by ROI (best/cheapest first):

1. **Transkun for piano source** — one-line addition to `TRANSCRIBERS` in `render_direct.py`. Beats current ByteDance path on clean piano recordings. `pip install transkun`, single-file CLI, MIT. Half day.

2. **mt3-infer for general polyphonic source** — `pip install mt3-infer[torch]`, wraps MT3/MR-MT3/YourMT3+ behind one API, returns standard MIDI. Fixes the "Basic Pitch drops notes" issue on non-piano audio. Half day.

3. **Source-separate then transcribe** — the community best-practice pattern. BS-Roformer splits mixes into piano/vocal/bass/drums stems, each stem routes to its best transcriber. See [mason369/music-to-midi](https://github.com/mason369/music-to-midi) for a 13-route reference impl. 2–3 days to integrate.

4. **Real-audio velocity envelope** — replace Basic Pitch's fake `127×amplitude` with an RMS-envelope derived from the source. Preserves dynamic contour. Doesn't apply if using Transkun/ByteDance (they emit real velocity). Half day.

5. **Sustain-pedal detection** — energy-based CC64 estimator for the transcribers that don't emit it. Feeds into `dp.apply_sustain` so the render matches source pedaling. Half day.

6. **Better SF2s** — Salamander Grand Piano V3 (real Yamaha C5, 800 MB), VSCO 2 Community Edition strings (9 GB, needs sfizz backend). Zero code changes once dropped in.

7. **Real impulse responses** — swap synthesized noise-decay reverb for real CC-BY IR (small hall, cathedral, plate). One hour.

8. **Pitch-bend / vibrato tracking** — per-note continuous f0 from source, emit as MIDI pitch-bend events. FluidSynth respects them. Recovers expression the transcriber quantizes away. 2 days.

9. **sfizz backend** — SFZ instrument support alongside SF2. Unlocks VSCO 2 CE and other high-quality SFZ libraries. 1 day.

10. **Optional ML re-processing pass** — if we ever build the performance-nuance ML model described in §10, run it as a *post-processor* on top of the direct render. Adds expression without replacing the timbre.

—

**Not doing**: retraining the mel-imitation model. Same architectural ceiling. If we return to ML for change-instrument, it needs a different formulation entirely (direct-audio conditioning + adversarial + codec-latent, not mel-imitation).

**Reference / current state**: as of this notebook the direct pipeline is the primary change-instrument path. Deprecated ML checkpoints stashed at [Ganeshveeer/retone-poly](https://huggingface.co/Ganeshveeer/retone-poly). Direct-pipeline sample renders (Bohemian × 29 instrument variants) at `~/Downloads/retone_direct_batch/`.